In [ ]:
# # NOTEBOOK NAME
# # RadarNetCDFsaverCompressed.ipynb
# # NOTEBOOK NAME

# OPENING IMPORTS
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr
import pandas as pd

from pathlib import Path      # used to play with pathnames to save 

import pyart

# SPECIAL METHOD TO IMPORT LEROI RADAR GRIDDING PACKAGE FROM LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks')
from leroi.leroi import *

from multiprocessing import Pool       # for parallelisation
from joblib import Parallel, delayed   # for parallelisation

In [ ]:
# CHAD CREATED FUNCTION BASED ON EXISTING LEROI INTERPOLATION


def process_radar_timestep(MinOfDay):
    """Process a single radar timestep."""
    try:
        # Calculate time indices
        houri = MinOfDay // 60
        mini = MinOfDay % 60
        RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00'
        RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6]
        print(f'Working on {RadarFileDatePrint} {RadarFileTimePrint}')
        
        # Read radar data
        RadarFolder = f'/scratch/v46/sg3241/tmp/UnzippedRadarFiles/{RadarIDno}/{RadarIDno}_{RadarFileDate}_ppi/'
        RadarFile = f'{RadarIDno}_{RadarFileDate}_{RadarFileTime}_ppi.nc'
        
        radar = pyart.io.read(RadarFolder + RadarFile, delay_field_loading=True)
        fields = list(radar.fields.keys())
        
        # Define grid
        grid_shape = (41, 301, 301)
        grid_limits = ((0.0, 20000.0), (-150000.0, 150000.0), (-150000.0, 150000.0))
        coords = [
            np.linspace(lower, upper, n)
            for (lower, upper), n in zip(grid_limits, grid_shape)
        ]
        
        # Interpolate and grid
        grid_fields = leroi_interp(
            radar,
            coords,
            field_names=fields,
            weight_type="Barnes",
            Rc=None,
            k=100,
            verbose=False  # Set to False for parallel runs
        )
        
        grid = build_pyart_grid(radar, grid_fields, grid_shape, grid_limits)
        xgrid = grid.to_xarray()
        
        # Save NetCDF
        NetCDFsaveFolder = f'/scratch/v46/sg3241/tmp/NetCDFs/CompressedRadarGrids/{RadarIDno}/{YYYY}/{MM}/{DD}/'
        NetCDFsaveFile = f'{RadarIDno}_{RadarFileDate}_{RadarFileTime}.nc'
        NetCDFsavePath = NetCDFsaveFolder + NetCDFsaveFile
        
        if not Path(NetCDFsaveFolder).exists():
            Path(NetCDFsaveFolder).mkdir(parents=True, exist_ok=True)
        
        encoding = {}
        for var in xgrid.data_vars:
            if xgrid[var].ndim < 2:
                continue
            encoding[var] = {
                "zlib": True,
                "complevel": 3,
                "fletcher32": True,
                "chunksizes": tuple(s // 2 for s in xgrid[var].shape),
            }
        
        xgrid.to_netcdf(NetCDFsavePath, format="NETCDF4", engine="netcdf4", encoding=encoding)
        return f"✓ Completed {RadarFileTimePrint}"
        
    except Exception as e:
        return f"✗ Failed {RadarFileTimePrint}: {str(e)}"

In [ ]:
# CHAD PARALLELISED VERSION
# DOES NOT UNZIP
# DOES NOT UNZIP
# DOES NOT UNZIP
# DOES NOT UNZIP
# DOES NOT UNZIP

# THIS WHOLE BLOCK CREATES AND STORES NET CDF FILES FOR GRIDDED RADAR DATA

# THIS BLOCK ALWAYS SAVES THE LOSSLESS COMPRESSED NETCDF FILES (OPTION 3)

# CHOOSE YOUR RADAR

# Radar Number Catalogue:
# Down The Coast YES Dual-Pol: 22 is Mackay,    106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marburg (near Bris)
# Down The Coast not Dual-Pol: 19 is Cairns,    8 is Gympie
# Down The Coast NO DOPPLER:   24 is Bowen,     23 is Gladstone (SPECIAL ELEVATION ANGLES)
# 0.8, 1.6, 2.4
# 3.6, 5.6
# 8.0, 11.5
# 16.0, 22.0, 32.0
#Inland
# Down inland YES Dual-Pol:    74 is Greenvale, 98 is Taroom,    108 is Towoomba
# Down inland not Dual-Pol:    78 is Weipa,     72 is Emerald
RadarIDno = '66' 

# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 12

for RadarDay in range(4,6+1):

# RadarDay   = 25
    # add leading zeros for strings
    YYYY = str(RadarYear).zfill(4)
    MM = str(RadarMonth).zfill(2)
    DD = str(RadarDay).zfill(2)
    
    RadarFileDate  = YYYY + MM + DD
    RadarFileDatePrint = YYYY + '-' + MM + '-' + DD
    
    if __name__ == '__main__':
        LoopStartTime = '00:00'
        LoopEndTime = '23:55'
        
        StartHour, StartMin = int(LoopStartTime.split(':')[0]), int(LoopStartTime.split(':')[1])
        EndHour, EndMin = int(LoopEndTime.split(':')[0]), int(LoopEndTime.split(':')[1])
        
        StartMinOfDay = StartHour * 60 + StartMin
        EndMinOfDay = EndHour * 60 + EndMin
        
        timesteps = list(range(StartMinOfDay, EndMinOfDay + 1, 5))
        
        # Run in parallel
        results = Parallel(n_jobs=4, verbose=10)(
            delayed(process_radar_timestep)(ts) for ts in timesteps
        )
        
        for result in results:
            print(result)

In [ ]:
# THIS WHOLE BLOCK CREATES AND STORES NET CDF FILES FOR GRIDDED RADAR DATA

# THIS BLOCK ALWAYS SAVES THE LOSSLESS COMPRESSED NETCDF FILES (OPTION 3)

# the reference number for the radar location
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)

# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 1
RadarDay   = 2

LoopStartTime = '01:25'
LoopEndTime   = '23:55'

# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)

RadarFileDate  = YYYY + MM + DD 

# THIS BLOCK TAKES A ZIPPED FOLDER FROM /G/DATA/ AND UNZIPS IT TO A FOLDER "nzippedRadarFiles" IN SCRATCH

# PATH NAMES
# ZippedFolder = '/g/data/rq0/level_1b/22/ppi/2024/'
ZippedFolder = '/g/data/rq0/level_1b/' + RadarIDno + '/ppi/' + str(RadarYear) + '/'
ZippedFile   =  RadarIDno + '_' + RadarFileDate + '_ppi.zip'

# place where the zipped file lives
ZippedPath = ZippedFolder + ZippedFile
# place to extract the files to
ExtractToDirectory = Path('/scratch/v46/sg3241/tmp/UnzippedRadarFiles/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_ppi/')

# Create the extraction folder (only if you haven't already)
if not Path(ExtractToDirectory).exists():
    ExtractToDirectory.mkdir(parents=True, exist_ok=True)

    # actually do the unzipping
    with zp.ZipFile(ZippedPath, 'r') as ZipReference:
        ZipReference.extractall(ExtractToDirectory)

    print('Done')
else:
    print('Unzipped Folder Already Exists')


# LOOP OVER EVERY 5 MIN PERIOD IN THE DAY
# Parse start and end times
StartHour, StartMin = int(LoopStartTime.split(':')[0]), int(LoopStartTime.split(':')[1])
EndHour, EndMin     =   int(LoopEndTime.split(':')[0]),   int(LoopEndTime.split(':')[1])

# Convert to total minutes for easy comparison
StartMinOfDay = StartHour * 60 + StartMin
EndMinOfDay = EndHour * 60 + EndMin

for MinOfDay in range(StartMinOfDay, EndMinOfDay + 1, 5):
    # find the old indicies with which this code was written (0-23 for hours, 0-11 for 5-minute periods within hours)
    houri = MinOfDay // 60
    mini  = MinOfDay % 60

    RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
    RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] # add a string of format hh:mm:ss for printing
    print('working on ' + RadarFileTimePrint)
    

    # READ IN THE DATA TO A PYART FILE
    
    # the path of where the UNZIPPED radar data now live
    RadarFolder = '/scratch/v46/sg3241/tmp/UnzippedRadarFiles/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_ppi/'
    RadarFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_ppi.nc'
    
    # open the radar data 
    radar = pyart.io.read(RadarFolder + RadarFile, delay_field_loading = True)  
    # radar = pyart.io.read(RadarFolder + RadarFile, delay_field_loading=False)
    
    # # RETRIEVE SOME FIELDS IN THE RADAR DATA
    # preferred_fields = ['corrected_reflectivity', 'corrected_velocity', 'corrected_differential_reflectivity',\
    #                     'corrected_cross_correlation_ratio', 'corrected_differential_phase', 'corrected_specific_differential_phase']
    
    # field = next((name for name in preferred_fields if name in radar.fields), next(iter(radar.fields)))
    # fields = [field]
    # field

    # Get ALL available fields from the radar
    fields = list(radar.fields.keys())
    print(f"Gridding fields: {fields}")

    # # Get only the corrected_reflectivity field
    # fields = 'corrected_reflectivity'
    # print(f"Gridding fields: {field}")
    
    #DEFINING THE CARTESIAN GRID SHAPE
    grid_shape = (41, 301, 301)
    grid_limits = ((0.0, 20000.0), (-150000.0, 150000.0), (-150000.0, 150000.0))
    coords = [
        np.linspace(lower, upper, n)
        for (lower, upper), n in zip(grid_limits, grid_shape) ]
    
    # SETS THE WEIGHTING AND CREATES A DICTIONARY OF FIELDS OR SOMETHING
    # I DON'T REALLY KNOW
    # ??????
    grid_fields = leroi_interp(
        radar,
        coords,
        field_names=fields,
        weight_type="Barnes",
        Rc=None,
        k=100,
        verbose=True )
    
    grid = build_pyart_grid(radar, grid_fields, grid_shape, grid_limits)
    # grid
    
    xgrid = grid.to_xarray() # change this loaded grid into an xarray data frame so I know how to work with it

    # save the converted gridded data as a re-usable NetCDF file
    NetCDFsaveFolder = '/scratch/v46/sg3241/tmp/NetCDFs/CompressedRadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/'
    NetCDFsaveFile   =  RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc'
    
    NetCDFsavePath = NetCDFsaveFolder + NetCDFsaveFile
    
    if not Path(NetCDFsaveFolder).exists():
        print('Creating Folder: ' + NetCDFsaveFolder)
        Path(NetCDFsaveFolder).mkdir(parents=True, exist_ok=True)

    encoding = {}
    for var in xgrid.data_vars:
        if xgrid[var].ndim < 2:
            continue  # skip 0D/1D if you don’t want to compress them
        encoding[var] = {
            "zlib": True,
            "complevel": 3,          # 0–9, higher = more compression, slower
            "fletcher32": True,      # optional checksum
            "chunksizes": tuple(s // 2 for s in xgrid[var].shape),
        }
    xgrid.to_netcdf(NetCDFsavePath, format="NETCDF4", engine="netcdf4", encoding=encoding)

In [ ]:
RadarIDno = '22'
YYYY = '2024'
MM = '02'
DD = '27'
RadarFileDate = YYYY + MM + DD
RadarFileTime = '031000'

NetCDFstoragePath = ('/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/' + \
                        RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')

NetCDFstoragePath_comp = ('/scratch/v46/sg3241/tmp/NetCDFs/CompressedRadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/' + \
                        RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')

# try to load in the netcdf file
xgrid = xr.open_dataset(NetCDFstoragePath)
xgrid_comp = xr.open_dataset(NetCDFstoragePath_comp)

In [ ]:
xgrid_comp['corrected_reflectivity'] = xgrid_comp['corrected_reflectivity'] +1


In [ ]:
np.nanmean(xgrid_comp['corrected_reflectivity'])

In [ ]:
# SAVE A PARTICULAR NET CDF FILE AS A COMPRESSED FILE

OutFilePath = '/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/22/2024/03/09/22_20240309_120000_CompressionLevel3.nc'
# Create all parent directories if they don't exist
Path(OutFilePath).parent.mkdir(parents=True, exist_ok=True)
encoding = {}
for var in xgrid.data_vars:
    if xgrid[var].ndim < 2:
        continue  # skip 0D/1D if you don’t want to compress them
    encoding[var] = {
        "zlib": True,
        "complevel": 3,          # 0–9, higher = more compression, slower
        "fletcher32": True,      # optional checksum
        "chunksizes": tuple(s // 2 for s in xgrid[var].shape),
    }
xgrid.to_netcdf(OutFilePath, format="NETCDF4", engine="netcdf4", encoding=encoding)


In [ ]:
# CHECK THE DIFFERENCES IN VALUES FOR THE COMPRESSED AND UNCOMPRESSED FILES

print("=" * 80)
print("COMPRESSION QUALITY ASSESSMENT")
print("=" * 80)

# Store results for summary
results = []

for var in xgrid.data_vars:
    # Skip coordinate variables and low-dimensional data
    if xgrid[var].ndim < 2:
        continue
    
    # Get flattened arrays
    original = np.array(xgrid[var]).ravel()
    compressed = np.array(xgrid_comp[var]).ravel()
    
    # Calculate differences
    diffs = compressed - original
    
    # Filter out NaN values
    valid_diffs = diffs[~np.isnan(diffs)]
    
    # Calculate statistics
    total_diff = np.sum(valid_diffs)
    mean_diff = np.mean(valid_diffs)
    max_diff = np.max(np.abs(valid_diffs))
    num_valid = len(valid_diffs)
    num_total = len(diffs)
    
    # Store results
    results.append({
        'Variable': var,
        'Total Difference': total_diff,
        'Mean Difference': mean_diff,
        'Max Abs Difference': max_diff,
        'Valid Values': num_valid,
        'Total Values': num_total,
    })
    
    # Print individual results
    print(f"\n{var}:")
    print(f"  Total Difference:      {total_diff:>15.6e}")
    print(f"  Mean Difference:       {mean_diff:>15.6e}")
    print(f"  Max Abs Difference:    {max_diff:>15.6e}")
    print(f"  Valid Values:          {num_valid:>15,} / {num_total:,}")

# Create summary table
print("\n" + "=" * 80)
print("SUMMARY TABLE")
print("=" * 80)

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

# Overall statistics
print("\n" + "=" * 80)
print("OVERALL STATISTICS")
print("=" * 80)
print(f"Total variables compared: {len(results)}")
print(f"Max total difference across all variables: {df_results['Total Difference'].abs().max():.6e}")
print(f"Mean of mean differences: {df_results['Mean Difference'].mean():.6e}")
